# Smart Lender - Loan Approval Prediction System
## Exploratory Data Analysis and Modeling
This notebook documents the data preprocessing and machine learning pipeline for predicting loan approvals using the official SkillWallet dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Load Official Dataset

In [ ]:
df = pd.read_csv('../dataset/loan_prediction.csv')
df = df.drop('Loan_ID', axis=1)
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of Loan Status
sns.countplot(x='Loan_Status', data=df)
plt.title('Loan Status Distribution')
plt.show()

In [ ]:
# Missing Values
df.isnull().sum()

## 3. Data Preprocessing
### 3.1 Handling Missing Values

In [ ]:
# Numerical: median
for col in ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Categorical: mode
for col in ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

### 3.2 Encoding Categorical Variables
Maintaining single source of truth for feature order.

In [ ]:
feature_columns = [
    'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
    'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term',
    'Credit_History', 'Property_Area'
]

encoders = {}
categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

le_target = LabelEncoder()
df['Loan_Status'] = le_target.fit_transform(df['Loan_Status'])

X = df[feature_columns]
y = df['Loan_Status']

### 3.3 Scaling and SMOTE

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

smote = SMOTE(random_state=42)
X_bal, y_bal = smote.fit_resample(X_scaled, y)

## 4. Train/Test Split
Using 0.33 test size as consistently documented.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_bal, y_bal, test_size=0.33, random_state=42)
print(f'Training size: {X_train.shape}, Test size: {X_test.shape}')

## 5. Model Training and Comparison

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

best_model = None
best_accuracy = 0

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'{name} Accuracy: {acc:.4f}')
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model

print(f'
Best Model: {best_model.__class__.__name__} with accuracy {best_accuracy:.4f}')

## 6. Export Models and Scaler

In [ ]:
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(encoders, '../models/encoders.pkl')
joblib.dump(feature_columns, '../models/feature_columns.pkl')
print('Artifacts saved successfully.')